# Hito 1 · Reproduced reference

**NextLap · IIT414W · Team Lab 8**

Reproduces the three references (training median, persistence, OLS) on the fixed validation rows.

- Target: `next_lap_s` (seconds), predicted at the end of lap *t*
- Train: rounds 1–7 (5,772 clean pairs) · Validation: rounds 8–9 (1,808 clean pairs)
- Preprocessing fitted on **training rows only** (sklearn Pipeline from the course helper)
- No held-out test data are read · `RANDOM_SEED = 414`

Adapted from the course script `probar_referencia_v1.py`, using the course helper `nextlap_reference_v1.py` (with attribution).

## 1. Setup and data location

In [1]:
import hashlib, json, os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

HITO_ROOT = Path.cwd()
DATA_DIR = Path(os.environ.get(
    "NEXTLAP_DATA", HITO_ROOT.parent / "W09_StudentPack_v2" / "base_NextLap")).resolve()
OUT_DIR = HITO_ROOT / "outputs"
RANDOM_SEED = 414

assert (DATA_DIR / "train_clean_v1.csv").is_file(), (
    f"Course data not found at {DATA_DIR}. Set NEXTLAP_DATA (see README.md).")
sys.path.insert(0, str(DATA_DIR))
import nextlap_reference_v1 as R  # course helper, used as released

np.random.seed(RANDOM_SEED)
print("Data folder found:", DATA_DIR.name)
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

Data folder found: base_NextLap
numpy 2.5.4 | pandas 3.1.1 | sklearn 1.9.1


## 2. Load data and check the fixed comparison

In [2]:
train = pd.read_csv(DATA_DIR / "train_clean_v1.csv")
val = pd.read_csv(DATA_DIR / "validation_clean_v1.csv")

assert len(train) == 5772 and len(val) == 1808
assert set(train["round"]) == set(range(1, 8)) and set(val["round"]) == {8, 9}
assert train.pair_id.is_unique and val.pair_id.is_unique
assert not set(train.pair_id) & set(val.pair_id)          # no pair in both splits

features = R.NUMERIC + R.CATEGORICAL
assert R.TARGET not in features and not any(c.startswith("next_") for c in features)  # no future inputs

print("Train rows:", len(train), "| Validation rows:", len(val))
print("Inputs:", features)
val.groupby("round").size().rename("validation_rows").to_frame()

Train rows: 5772 | Validation rows: 1808
Inputs: ['current_lap_s', 'recent_mean_s', 'current_lap_number', 'current_tyre_life', 'current_compound', 'current_team']


,validation_rows
round,
8,1066
9,742


## 3. Fit the three references

In [3]:
def mae(y, p):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p))))

ols = R.make_model("OLS").fit(train[features], train[R.TARGET])   # imputer/scaler/encoder fitted on train only

preds = {
    "train_median": np.full(len(val), train[R.TARGET].median()),
    "persistence": val.current_lap_s.to_numpy(),                 # next lap = current lap
    "OLS_reference": ols.predict(val[features]),
}
assert all(np.isfinite(p).all() for p in preds.values())

## 4. Validation MAE (seconds), overall and by race

In [4]:
rows = []
for name, pred in preds.items():
    for label, mask in [("all_validation", np.ones(len(val), dtype=bool)),
                        ("round_8", val["round"].eq(8).to_numpy()),
                        ("round_9", val["round"].eq(9).to_numpy())]:
        rows.append({"model": name, "population": label, "n": int(mask.sum()),
                     "MAE_s": mae(val.loc[mask, R.TARGET], pred[mask])})
metrics = pd.DataFrame(rows)
metrics.pivot(index="model", columns="population", values="MAE_s").round(4)

population,all_validation,round_8,round_9
model,,,
OLS_reference,0.3867,0.3736,0.4056
persistence,0.4093,0.4005,0.4219
train_median,11.3761,11.7158,10.8881


**Reading the table.** One median for every lap is very poor because lap times differ a lot between circuits.
Repeating the current lap time already brings the error to about 0.41 s, and OLS improves on that by only about 0.023 s.
All three models are scored on the same 1,808 `pair_id`s.

## 5. Save outputs

In [5]:
OUT_DIR.mkdir(exist_ok=True)
metrics.to_csv(OUT_DIR / "reference_metrics.csv", index=False)
pd.DataFrame({"pair_id": val.pair_id, "round": val["round"], "y_true_s": val[R.TARGET],
              **preds}).to_csv(OUT_DIR / "validation_reference_predictions.csv", index=False)
(OUT_DIR / "reference_run.json").write_text(json.dumps({
    "train_rows": len(train), "validation_rows": len(val), "test_used": False,
    "seed": RANDOM_SEED, "features": features, "target": R.TARGET, "metric": "MAE (s)",
    "validation_pair_id_sha256": hashlib.sha256("\n".join(sorted(val.pair_id)).encode()).hexdigest(),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
    "data_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in
                    [DATA_DIR / "train_clean_v1.csv", DATA_DIR / "validation_clean_v1.csv",
                     DATA_DIR / "nextlap_reference_v1.py"]},
    "results": rows,
}, indent=2), encoding="utf-8")
print("Written:", sorted(p.name for p in OUT_DIR.iterdir()))
print("No test data used.")

Written: ['reference_metrics.csv', 'reference_run.json', 'validation_reference_predictions.csv']
No test data used.
